In [1]:
!pip uninstall -y tensorflow
!pip install mediapipe==0.10.14 protobuf==4.25.9

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.2

In [2]:
import mediapipe as mp
print(mp.__version__)


0.10.14


In [3]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import os

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

Uzyskiwanie danych:

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
data_dir = "/content/drive/MyDrive/1_DATASET_TENIS"
os.listdir(data_dir)


['waiting', 'backhand', 'forehand', 'serwis']

Inicjalizacja mediapipe

In [7]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False, #może spróbować na true???
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

In [8]:
SELECT_LANDMARKS = [
    11, 12,  # barki
    13, 14,  # łokcie
    15, 16,  # nadgarstki
    23, 24,  # biodra
    25, 26,  # kolana
    27, 28   # kostki
]

funkcja ekstrakcji jedynie potrzebnych landmarków

In [9]:
def extract_simplified_landmarks(frame):
    """Zwraca wektor 48-elementowy: x,y,z,visibility dla wybranych punktów"""
    results = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if not results.pose_landmarks:
        return None  # brak detekcji

    lm = []
    for idx in SELECT_LANDMARKS:
        p = results.pose_landmarks.landmark[idx]
        lm.extend([p.x, p.y, p.z, p.visibility])
    lm = np.array(lm).reshape(len(SELECT_LANDMARKS), 4)

    # centrowanie względem środka bioder
    hip_center = lm[6:8, :2].mean(axis=0)  # index 6 i 7 = lewe i prawe biodro
    lm[:, :2] -= hip_center

    # skalowanie względem szerokości barków
    shoulder_distance = np.linalg.norm(lm[0, :2] - lm[1, :2])  # barki
    lm[:, :2] /= shoulder_distance + 1e-6  # uniknięcie dzielenia przez 0

    return lm.flatten()  # finalnie shape = (48,)

In [10]:
def extract_all_landmarks_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames_landmarks = []
    last_valid = None

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        landmarks = extract_simplified_landmarks(frame)

        if landmarks is not None:
            last_valid = landmarks
        else:
            if last_valid is not None:
                landmarks = last_valid
            else:
                continue

        frames_landmarks.append(landmarks)

    cap.release()

    return frames_landmarks

Dtataset dla Pytorcha

In [11]:
class PoseDataset(Dataset):
    def __init__(self, root_dir, seq_len=20, limit_per_class=70):
        self.samples = [] # Nazwa musi być zgodna z __getitem__
        self.labels = []
        self.seq_len = seq_len

        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(class_names)}

        for cls in class_names:
            class_path = os.path.join(root_dir, cls)
            if not os.path.isdir(class_path):
                continue

            count = 0
            print(f"Rozpoczynam przetwarzanie klasy: {cls}...")

            for file in os.listdir(class_path):
                if count >= limit_per_class:
                    break

                if file.endswith(".mp4"):
                    video_path = os.path.join(class_path, file)

                    # Tutaj następuje faktyczne przetwarzanie przez MediaPipe
                    all_landmarks = extract_all_landmarks_from_video(video_path)

                    if len(all_landmarks) < seq_len:
                        continue

                    # Sliding window (okno przesuwne)
                    for i in range(0, len(all_landmarks) - seq_len + 1):
                        sequence = all_landmarks[i:i+seq_len]
                        self.samples.append(sequence)
                        self.labels.append(self.class_to_idx[cls])

                    count += 1
                    if count % 10 == 0:
                        print(f"  Przetworzono {count}/{limit_per_class} filmów...")

            print(f"Zakończono klasę {cls}. Łącznie sekwencji: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # Ta metoda teraz zadziała, bo self.samples już istnieje
        sequence = np.array(self.samples[idx])
        label = self.labels[idx]

        return (
            torch.tensor(sequence, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long)
        )

In [13]:
seq_len = 20
batch_size = 8

full_dataset = PoseDataset(data_dir, seq_len=seq_len)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

Rozpoczynam przetwarzanie klasy: backhand...
  Przetworzono 10/70 filmów...
  Przetworzono 20/70 filmów...
  Przetworzono 30/70 filmów...
  Przetworzono 40/70 filmów...
  Przetworzono 50/70 filmów...
  Przetworzono 60/70 filmów...
  Przetworzono 70/70 filmów...
Zakończono klasę backhand. Łącznie sekwencji: 5300
Rozpoczynam przetwarzanie klasy: forehand...
  Przetworzono 10/70 filmów...
  Przetworzono 20/70 filmów...
  Przetworzono 30/70 filmów...
  Przetworzono 40/70 filmów...
  Przetworzono 50/70 filmów...
  Przetworzono 60/70 filmów...
  Przetworzono 70/70 filmów...
Zakończono klasę forehand. Łącznie sekwencji: 12035
Rozpoczynam przetwarzanie klasy: serwis...
  Przetworzono 10/70 filmów...
  Przetworzono 20/70 filmów...
  Przetworzono 30/70 filmów...
  Przetworzono 40/70 filmów...
  Przetworzono 50/70 filmów...
  Przetworzono 60/70 filmów...
  Przetworzono 70/70 filmów...
Zakończono klasę serwis. Łącznie sekwencji: 23927
Rozpoczynam przetwarzanie klasy: waiting...
  Przetworzono 10/7

Architektura CN1D

In [14]:
class PoseCNN1D(nn.Module):
    def __init__(self, input_size=48, num_classes=4):
        super().__init__()

        self.conv1 = nn.Conv1d(input_size, 128, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

Trening

In [15]:
model = PoseCNN1D(input_size=48, num_classes=4).to(device)

In [16]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(),
                             lr=0.001)

In [17]:
epochs = 30

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            val_loss += loss.item()

            # dokładność
            predicted = torch.argmax(y_pred, dim=1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total

    print(f"Train Loss: {train_loss:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch 1/30
Train Loss: 0.4291, Val Loss: 0.2614, Val Acc: 0.9456
Epoch 2/30
Train Loss: 0.1721, Val Loss: 0.0957, Val Acc: 0.9731
Epoch 3/30
Train Loss: 0.0875, Val Loss: 0.0710, Val Acc: 0.9827
Epoch 4/30
Train Loss: 0.0770, Val Loss: 0.0527, Val Acc: 0.9817
Epoch 5/30
Train Loss: 0.0481, Val Loss: 0.0371, Val Acc: 0.9891
Epoch 6/30
Train Loss: 0.0499, Val Loss: 0.0331, Val Acc: 0.9936
Epoch 7/30
Train Loss: 0.0412, Val Loss: 0.0289, Val Acc: 0.9879
Epoch 8/30
Train Loss: 0.0302, Val Loss: 0.3871, Val Acc: 0.9520
Epoch 9/30
Train Loss: 0.0558, Val Loss: 0.0094, Val Acc: 0.9984
Epoch 10/30
Train Loss: 0.0312, Val Loss: 0.0133, Val Acc: 0.9966
Epoch 11/30
Train Loss: 0.0239, Val Loss: 0.3486, Val Acc: 0.9619
Epoch 12/30
Train Loss: 0.0256, Val Loss: 0.0091, Val Acc: 0.9985
Epoch 13/30
Train Loss: 0.0171, Val Loss: 0.0086, Val Acc: 0.9982
Epoch 14/30
Train Loss: 0.0316, Val Loss: 0.0343, Val Acc: 0.9981
Epoch 15/30
Train Loss: 0.0272, Val Loss: 0.0204, Val Acc: 0.9984
Epoch 16/30
Train L

Test